# Step 16a — bundles of three other papers for the tabular foundation models, with conventional baselines

Step 8 ran TabPFN and TabICL on the variables of Dang et al. (2024). This step prepares the same test for three other NHANES depression papers, each on its own cohort and variables:

| bundle | paper | cohort (ours / paper) | variables | paper's split | paper's best AUC |
|---|---|---|---|---|---|
| `chen2025_living_alone` | Chen et al. 2025, *Front Public Health* 13:1472050, adults living alone | 2,787, 278 positive / 2,642, 279 | 74 candidates (the paper lists 72–73); the paper's 30 selected | 80/20 | 0.85 stacked ensemble; 0.82 RF, 0.81 CatBoost / XGBoost |
| `jin2025_pfas` | Jin et al. 2025, *J Affect Disord* 391:119976, serum PFAS | 9,395, 816 positive / 9,074, 786 | 11 covariates + 7 PFAS | 80/20 | 0.75 CatBoost |
| `shangguan2026_inactive` | ShangGuan et al. 2026, *J Affect Disord* 394:120555, physically inactive adults | development 9,856 / 8,643; three validation sets | 32 candidates; the paper's 7 selected | 70/30 + validation sets | 0.769 LR (validation 0.736–0.794) |

The cohorts come from our reproductions of these papers (built from the papers' descriptions; the differences in size are listed in their `config.json`). Price et al. 2024 (wearable movement data) is not included: its minute-level accelerometer features have not been built.

**A bundle** (`$NHANES_WORK/modeling/step16_bundles/<bundle>/`) holds everything the GPU notebook needs:
- `data.csv.gz`: `SEQN`, `part` (the paper's split: train / test, and val1–val3 for ShangGuan), `fold` (10-fold `StratifiedKFold(10, shuffle, seed 42)` over the train + test rows; −1 for validation-only rows), `depression` (PHQ-9 ≥ 10), `phq9` (total), then the variables.
- `columns.json`: categorical and continuous variables, and the feature sets (`all`, and `paper` = the paper's selected variables where it selected).
- `baselines/`: predictions of conventional models on exactly the same tasks, computed here on CPU.

**Tasks**, the same for baselines and foundation models:
- `cv0`–`cv9`: train on 9 folds, predict the 10th. Stable, and can be compared fold by fold.
- `paper`: train on the paper's training part, predict its test part (and validation sets). Comparable with the paper's number, but a single split; for Chen 2025 the test part has only 56 positives.

**Baselines:** logistic regression (L2), random forest, XGBoost and CatBoost with fixed, ordinary settings and no tuning (the foundation models are not tuned either). They stand for "conventional models on the same data", and are not the papers' exact pipelines: no SMOTE (Chen), no missForest (ShangGuan: median imputation inside the training rows; trees get the missing values as they are), and Chen's stacked neural network is left out because with the settings in the paper's supplement it outputs a constant.

**Inputs** (with `RUN`): `CHEN2025_COHORT`, `JIN2025_COHORT` (the `cohort.csv` of the reproductions) and `SHANGGUAN2026_RUN` (the run folder with `dev_raw.csv` and `val*_raw.csv`).

`RUN = True` writes the bundles and baselines (a few minutes, CPU); `RUN = False` only shows the saved ones.

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "nhanes" / "paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("run this notebook from inside the repository")
sys.path.insert(0, str(ROOT))
from nhanes.paths import RESULTS, THREADS  # noqa: E402

RUN = False  # True: write the bundles and the baselines
RUN = RUN or os.environ.get("NHANES_RUN") == "1"
OUT = RESULTS / "step16_bundles"
SEED = 42
BASELINES = ["logreg", "rf", "xgboost", "catboost"]

CHEN_TOP30 = ["DMDMARTL", "RIDAGEYR", "IND235", "INDFMMPI", "HEI2020_ALL", "HEI2020_DAIRY", "HEI2020_ADDEDSUGAR",
              "HEI2020_FATTYACID", "HEI2020_FRT", "HEI2020_GREENNBEAN", "HEI2020_SODIUM", "HEI2020_REFINEDGRAIN",
              "HEI2020_SATFAT", "HEI2020_SEAPLANTPRO", "HEI2020_TOTALFRT", "HEI2020_TOTALPRO", "HEI2020_VEG",
              "HEI2020_WHOLEGRAIN", "LBXCOT", "PAD680", "TPA", "SLD012", "SLQ050", "BMXBMI", "RXDDAYS", "RXDCOUNT",
              "PFQ054", "MCQ160K", "KIQ046", "HSD010"]
CHEN_CONT = ["BMXBMI", "INDFMMPI", "LBXCOT", "PAD680", "RIDAGEYR", "RPA", "TPA", "WPA", "SLD012", "RXDDAYS",
             "RXDCOUNT"]  # RXDCOUNT (number of drugs) is a count; the reproduction listed it as categorical
JIN_CAT = ["gender", "race", "education", "marital", "smoke", "alcohol", "hypertension", "diabetes"]
JIN_FEATURES = ["age", "gender", "race", "education", "marital", "PIR", "BMI", "smoke", "alcohol", "hypertension",
                "diabetes", "MPAH", "PFDE", "PFHxS", "PFNA", "PFOA", "PFOS", "PFUA"]
SG_CANDIDATES = ["AGE", "SEX", "RACE", "MARITAL", "EDU", "PIR", "SMOKE", "ALCOHOL", "SLEEP_TIME", "SLEEP_DISORDER",
                 "HYPERTENSION", "DIABETES", "HYPERLIPIDEMIA", "CVD", "ASCVD", "STROKE", "CANCER", "BMI", "WAIST",
                 "BRI", "SBP", "DBP", "TG", "TOTAL_C", "HDL_C", "UACR", "EGFR", "DII", "NLR", "MLR", "SIRI", "SII"]
SG_CAT = ["SEX", "RACE", "MARITAL", "EDU", "SMOKE", "ALCOHOL", "SLEEP_DISORDER", "HYPERTENSION", "DIABETES",
          "HYPERLIPIDEMIA", "CVD", "ASCVD", "STROKE", "CANCER"]
SG_PAPER7 = ["SLEEP_DISORDER", "PIR", "WAIST", "NLR", "SEX", "SII", "AGE"]

## Run: bundles

In [2]:
def add_folds(d):
    """10-fold ids over the train + test rows (-1 elsewhere)."""
    from sklearn.model_selection import StratifiedKFold
    d["fold"] = -1
    cv = np.flatnonzero(d.part.isin(["train", "test"]))
    for k, (_, te) in enumerate(StratifiedKFold(10, shuffle=True, random_state=SEED).split(cv, d.depression.iloc[cv])):
        d.iloc[cv[te], d.columns.get_loc("fold")] = k
    return d


def split_80_20(y):
    from sklearn.model_selection import train_test_split
    tr, _ = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)
    return np.where(np.isin(np.arange(len(y)), tr), "train", "test")


def write_bundle(name, d, feats, cat, sets, info):
    d = add_folds(d)[["SEQN", "part", "fold", "depression", "phq9"] + feats]
    assert d.phq9.notna().all() and ((d.phq9 >= 10).astype(int) == d.depression).all()
    (OUT / name).mkdir(parents=True, exist_ok=True)
    d.to_csv(OUT / name / "data.csv.gz", index=False)
    (OUT / name / "columns.json").write_text(json.dumps(
        {**info, "categorical": [c for c in feats if c in cat], "continuous": [c for c in feats if c not in cat],
         "feature_sets": sets}, indent=1))


if RUN:
    c = pd.read_csv(os.environ["CHEN2025_COHORT"]).rename(columns={"MDD": "depression", "PHQ9": "phq9"})
    feats = [x for x in c if x not in ("SEQN", "cycle", "depression", "phq9")]
    assert set(CHEN_TOP30) <= set(feats)
    c["part"] = split_80_20(c.depression)
    write_bundle("chen2025_living_alone", c, feats, [x for x in feats if x not in CHEN_CONT and not x.startswith("HEI")],
                 {"all": feats, "paper": CHEN_TOP30},
                 {"paper": "Chen et al. 2025, Front Public Health 13:1472050 (adults living alone)",
                  "reported_auc": {"stacked ensemble": 0.85, "RF": 0.82, "CatBoost": 0.81, "XGBoost": 0.81}})

    j = pd.read_csv(os.environ["JIN2025_COHORT"]).rename(columns={"PHQ9": "phq9"})
    j["part"] = split_80_20(j.depression)
    write_bundle("jin2025_pfas", j, JIN_FEATURES, JIN_CAT, {"all": JIN_FEATURES},
                 {"paper": "Jin et al. 2025, J Affect Disord 391:119976 (serum PFAS)",
                  "reported_auc": {"CatBoost": 0.75}})

    run = Path(os.environ["SHANGGUAN2026_RUN"])
    s = pd.read_csv(run / "dev_raw.csv").rename(columns={"split": "part"})
    vals = [pd.read_csv(run / f"{v}_raw.csv").assign(part=v.split("_")[0])
            for v in ["val1_2005_2006", "val2_2015_2016", "val3_2019_2020"]]
    s = pd.concat([s, *vals], ignore_index=True).rename(columns={"DEPRESSION": "depression", "PHQ9": "phq9"})
    write_bundle("shangguan2026_inactive", s, SG_CANDIDATES, SG_CAT, {"all": SG_CANDIDATES, "paper": SG_PAPER7},
                 {"paper": "ShangGuan et al. 2026, J Affect Disord 394:120555 (physically inactive adults)",
                  "reported_auc": {"LR": 0.769},
                  "note": "val1 = 2005-2006, val2 = 2015-2016, val3 = 2017-March 2020 files; val3 contains the "
                          "2017-2018 participants of the development set under new ids, so it is optimistic"})

## Run: conventional baselines on the same tasks

In [3]:
def tasks(d):
    """(name, training rows, rows to predict), the same for every model."""
    out = [(f"cv{k}", np.flatnonzero((d.fold >= 0) & (d.fold != k)), np.flatnonzero(d.fold == k)) for k in range(10)]
    return out + [("paper", np.flatnonzero(d.part == "train"), np.flatnonzero(d.part != "train"))]


def fit_predict(model, Xtr, ytr, Xte, cat):
    from catboost import CatBoostClassifier
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier

    num = [c for c in Xtr if c not in cat]
    as_level = lambda X: X[cat].astype("float").fillna(-99).astype(int).astype(str)  # noqa: E731  missing = own level
    if model == "catboost":
        A, B = Xtr.copy(), Xte.copy()
        A[cat], B[cat] = as_level(A), as_level(B)
        m = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=4, cat_features=cat, random_seed=SEED,
                               thread_count=THREADS, verbose=False, allow_writing_files=False)
        return m.fit(A, ytr).predict_proba(B)[:, 1]
    A, B = Xtr.copy(), Xte.copy()
    A[cat], B[cat] = as_level(A), as_level(B)
    onehot = ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)
    if model == "xgboost":  # missing continuous values go to the trees as they are
        ct = ColumnTransformer([("num", "passthrough", num), onehot])
        m = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                          tree_method="hist", n_jobs=THREADS, random_state=SEED)
    else:
        ct = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num), onehot])
        m = (LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000) if model == "logreg" else
             RandomForestClassifier(500, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=THREADS,
                                    random_state=SEED))
    return m.fit(ct.fit_transform(A), ytr).predict_proba(ct.transform(B))[:, 1]


if RUN:
    for bundle in sorted(p.name for p in OUT.iterdir() if (p / "columns.json").exists()):
        d = pd.read_csv(OUT / bundle / "data.csv.gz")
        cols = json.loads((OUT / bundle / "columns.json").read_text())
        for fs, feats in cols["feature_sets"].items():
            cat = [c for c in feats if c in cols["categorical"]]
            for model in BASELINES:
                dest = OUT / bundle / "baselines" / fs / model
                dest.mkdir(parents=True, exist_ok=True)
                for name, tr, te in tasks(d):
                    score = fit_predict(model, d.iloc[tr][feats], d.depression.iloc[tr], d.iloc[te][feats], cat)
                    d.iloc[te][["SEQN", "part", "fold", "depression", "phq9"]].assign(score=score).to_csv(
                        dest / f"{name}.csv", index=False)
            print(bundle, fs, "baselines done", flush=True)

## The bundles

In [4]:
from sklearn.metrics import roc_auc_score  # noqa: E402

rows, base = [], []
for bundle in sorted(p.name for p in OUT.iterdir() if (p / "columns.json").exists()):
    d = pd.read_csv(OUT / bundle / "data.csv.gz")
    cols = json.loads((OUT / bundle / "columns.json").read_text())
    for part, g in d.groupby("part", sort=False):
        rows.append({"bundle": bundle, "part": part, "people": len(g), "PHQ-9 ≥ 10": int(g.depression.sum()),
                     "rate": g.depression.mean(), "in the 10 folds": int((g.fold >= 0).sum())})
    print(f"{bundle}: {cols['paper']}; {len(cols['categorical'])} categorical + {len(cols['continuous'])} continuous; "
          f"feature sets {({k: len(v) for k, v in cols['feature_sets'].items()})}; "
          f"missing values {int(d[cols['categorical'] + cols['continuous']].isna().sum().sum()):,}")
    for path in sorted((OUT / bundle / "baselines").glob("*/*")):
        cv = pd.concat([pd.read_csv(f) for f in path.glob("cv*.csv")])
        per_fold = cv.groupby("fold").apply(lambda g: roc_auc_score(g.depression, g.score))
        paper = pd.read_csv(path / "paper.csv")
        row = {"bundle": bundle, "features": path.parent.name, "model": path.name,
               "10-fold AUROC": per_fold.mean(), "sd": per_fold.std()}
        row.update({f"paper split: {p}": roc_auc_score(g.depression, g.score) for p, g in paper.groupby("part")})
        base.append(row)
display(pd.DataFrame(rows).set_index(["bundle", "part"]).round(3))

chen2025_living_alone: Chen et al. 2025, Front Public Health 13:1472050 (adults living alone); 49 categorical + 25 continuous; feature sets {'all': 74, 'paper': 30}; missing values 0


jin2025_pfas: Jin et al. 2025, J Affect Disord 391:119976 (serum PFAS); 8 categorical + 10 continuous; feature sets {'all': 18}; missing values 0


shangguan2026_inactive: ShangGuan et al. 2026, J Affect Disord 394:120555 (physically inactive adults); 14 categorical + 18 continuous; feature sets {'all': 32, 'paper': 7}; missing values 22,480


people  PHQ-9 ≥ 10   rate  in the 10 folds
bundle                 part                                             
chen2025_living_alone  train    2229         222  0.100             2229
                       test      558          56  0.100              558
jin2025_pfas           train    7516         653  0.087             7516
                       test     1879         163  0.087             1879
shangguan2026_inactive train    6899         861  0.125             6899
                       test     2957         369  0.125             2957
                       val1     2170         164  0.076                0
                       val2     1915         187  0.098                0
                       val3     2869         296  0.103                0

Conventional baselines (fixed settings, no tuning). The papers' own numbers are in the table at the top.

In [5]:
display(pd.DataFrame(base).set_index(["bundle", "features", "model"]).round(3).fillna(""))

10-fold AUROC     sd  \
bundle                 features model                            
chen2025_living_alone  all      catboost          0.819  0.034   
                                logreg            0.799  0.032   
                                rf                0.816  0.035   
                                xgboost           0.810  0.030   
                       paper    catboost          0.814  0.033   
                                logreg            0.809  0.030   
                                rf                0.806  0.045   
                                xgboost           0.800  0.039   
jin2025_pfas           all      catboost          0.727  0.030   
                                logreg            0.720  0.033   
                                rf                0.724  0.031   
                                xgboost           0.722  0.030   
shangguan2026_inactive all      catboost          0.777  0.019   
                                logreg            0.765  0.019   
                                rf                0.778  0.021   
                                xgboost           0.779  0.020   
                       paper    catboost          0.722  0.018   
                                logreg            0.717  0.018   
                                rf                0.713  0.020   
                                xgboost           0.725  0.016   

                                          paper split: test paper split: val1  \
bundle                 features model                                           
chen2025_living_alone  all      catboost              0.784                     
                                logreg                0.777                     
                                rf                    0.768                     
                                xgboost               0.779                     
                       paper    catboost              0.758                     
                                logreg                0.765                     
                                rf                    0.732                     
                                xgboost               0.744                     
jin2025_pfas           all      catboost              0.750                     
                                logreg                0.740                     
                                rf                    0.746                     
                                xgboost               0.738                     
shangguan2026_inactive all      catboost              0.771              0.75   
                                logreg                0.766             0.732   
                                rf                    0.773              0.74   
                                xgboost               0.763             0.744   
                       paper    catboost              0.722             0.694   
                                logreg                0.718             0.682   
                                rf                    0.705             0.663   
                                xgboost               0.721              0.68   

                                         paper split: val2 paper split: val3  
bundle                 features model                                         
chen2025_living_alone  all      catboost                                      
                                logreg                                        
                                rf                                            
                                xgboost                                       
                       paper    catboost                                      
                                logreg                                        
                                rf                                            
                                xgboost                                       
jin2025_pfas    